# Elastic Chunking Demo

## Install Requirements

In [ ]:
! pip install -q -U -r requirements.txt

## Create Elastic Serverless Environment

In [ ]:
%%bash
echo "--- Initializing ---"
terraform -chdir=terraform init -upgrade -input=false > /dev/null && echo "Done."
echo "--- Applying Changes ---"
terraform -chdir=terraform apply -auto-approve > /dev/null && echo "Done."

echo "--- Exporting Environment Variables ---"
cat > .env << EOF
ELASTIC_CLOUD_API_KEY=$(terraform -chdir=terraform output -raw elastic_cloud_api_key)
ELASTIC_CLOUD_ID=$(terraform -chdir=terraform output -raw elastic_cloud_id)
EOF
echo "Done."

## Create Elastic Client & Load Document

In [ ]:
import os
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv(override=True)

es = Elasticsearch(
    cloud_id=os.environ["ELASTIC_CLOUD_ID"],
    api_key=os.environ["ELASTIC_CLOUD_API_KEY"]
)
print(f'Client connected: {es.ping()}')

with open("assets/sample_doc.md", "r") as f:
    doc_content = f.read()
print(f"Document word count: {len(doc_content.split())} words")

## Scenario 1: Semantic Text — Automatic Chunking

In [ ]:
S1_INDEX = "scenario1"
scenario1_mapping = {
    "mappings": {
        "properties": {
            "content": { 
                "type": "semantic_text",
                 "chunking_settings": {
                     "strategy": "recursive",
                     "max_chunk_size": 200,
                     "separator_group": "markdown"
                 } 
            }
        }
    }
}
es.options(ignore_status=[404]).indices.delete(index=S1_INDEX)
es.indices.create(index=S1_INDEX, body=scenario1_mapping)
es.index(index=S1_INDEX, id=1, document={"content": doc_content})
es.indices.refresh(index=S1_INDEX)

response = es.search(
    index=S1_INDEX,
    body={
        "query": { "ids": { "values": ["1"] } },
        "fields": [{ "field": "content", "format": "chunks" }]
    },
)
stored_chunks = response["hits"]["hits"][0]["fields"]["content"]
print(f"ES auto-chunked into {len(stored_chunks)} chunks:")
for i, c in enumerate(stored_chunks):
    print(f"  Chunk {i}: {len(c.split())} words — {c[:60].replace('\n', ' ')}...")
print("-" * 40)

QUERY = "How do distributed systems handle failures and recovery?"
response = es.search(
    index=S1_INDEX,
    body={
        "query": {
            "semantic": {
                "field": "content",
                "query": QUERY,
            }
        },
        "_source": False,
        "highlight": {
            "fields": {
                "content": {
                    "type": "semantic",
                    "number_of_fragments": 5,
                    "order": "score",
                }
            }
        },
        "size": 5
    },
)

print(f"\nQuery: {QUERY}")
print("Search Results (with semantic highlighting, in order or relevance)")
for hit in response["hits"]["hits"]:
    print(f"  Score: {hit['_score']:.4f}")
    for i, chunk in enumerate(hit["highlight"]["content"]):
        print(f"    Chunk:{chunk[:100].replace(chr(10), ' ').strip()}...")
    print(f"  ---")

## Scenario 2: Semantic Text — Pre-chunked Arrays

In [ ]:
import re

S2_INDEX = "scenario2"
scenario2_mapping = {
    "mappings": {
        "properties": {
            "content": { 
                "type": "semantic_text",
                 "chunking_settings": {
                     "strategy": "none"
                 } 
            }
        }
    }
}
chunks = re.split(r'(?=^#{1,3} )', doc_content, flags=re.MULTILINE)
chunks = [c.strip() for c in chunks if c.strip()]
print(f"Chunk count: {len(chunks)}")

for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {len(chunk.split())} words — {chunk[:60].replace(chr(10), ' ')}...")

es.options(ignore_status=[404]).indices.delete(index=S2_INDEX)
es.indices.create(index=S2_INDEX, body=scenario2_mapping)
es.index(index=S2_INDEX, document={"content": chunks})
es.indices.refresh(index=S2_INDEX)

QUERY = "How do distributed systems handle failures and recovery?"
response = es.search(
    index=S2_INDEX,
    body={
        "query": {
            "semantic": {
                "field": "content",
                "query": QUERY,
            }
        },
        "_source": False,
        "highlight": {
            "fields": {
                "content": {
                    "type": "semantic",
                    "number_of_fragments": 5,
                    "order": "score"
                }
            }
        },
        "size": 5
    },
)

print("-" * 40)
print(f"\nQuery: {QUERY}")
print("Search Results (with semantic highlighting)")
for hit in response["hits"]["hits"]:
    print(f"  Score: {hit['_score']:.4f}")
    for i, chunk in enumerate(hit["highlight"]["content"]):
        print(f"    Chunk: {chunk[:100].replace(chr(10), ' ')}...")
    print(f"  ---")

## Scenario 3: External Chunking — One Document Per Chunk

In [ ]:
from semchunk import chunk

EMBEDDING_MODEL = ".jina-embeddings-v5-text-small"
token_counter = lambda text: len(text.split())

S3_INDEX = "scenario3"
scenario3_mapping = {
    "mappings": {
        "properties": {
            "parent_id": { "type": "keyword" },
            "chunk_index": { "type": "integer" },
            "chunk_text": { "type": "text" },
            "embedding": {
                "type": "dense_vector",
                "dims": 1024,
                "index": True,
                "similarity": "cosine",
                "index_options": { "type": "int8_hnsw" }
            }
        }
    }
}

chunks = chunk(doc_content, chunk_size=200, token_counter=token_counter)
print(f"Chunk count: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: {len(c.split())} words — {c[:60].replace(chr(10), ' ')}...")
print("-" * 40)

es.options(ignore_status=[404]).indices.delete(index=S3_INDEX)
es.indices.create(index=S3_INDEX, body=scenario3_mapping)

embeddings = []
for batch_start in range(0, len(chunks), 16):
    batch = chunks[batch_start:batch_start + 16]
    result = es.inference.inference(
        inference_id=EMBEDDING_MODEL,
        task_type="text_embedding",
        body={"input": batch}
    )
    embeddings.extend(item["embedding"] for item in result["text_embedding"])

for i, (c, emb) in enumerate(zip(chunks, embeddings)):
    es.index(index=S3_INDEX, document={
        "parent_id": "sample_doc",
        "chunk_index": i,
        "chunk_text": c,
        "embedding": emb
    })
es.indices.refresh(index=S3_INDEX)

QUERY = "How do distributed systems handle failures and recovery?"
query_result = es.inference.inference(
    inference_id=EMBEDDING_MODEL,
    task_type="text_embedding",
    body={"input": [QUERY]}
)
query_embedding = query_result["text_embedding"][0]["embedding"]

response = es.search(
    index=S3_INDEX,
    body={
        "knn": {
            "field": "embedding",
            "query_vector": query_embedding,
            "k": 5,
            "num_candidates": 20
        },
        "size": 5
    }
)

print(f"\nQuery: {QUERY}")
print("Search Results")
for hit in response["hits"]["hits"]:
    print(f"  Score: {hit['_score']:.4f}  Chunk #: {hit['_source']['chunk_index']}")
    print(f"  Chunk: {hit['_source']['chunk_text'][:60].replace(chr(10), ' ')}...")
    print(f"  ---")

## Scenario 4: External Chunking — Hybrid Search with Reranking



In [ ]:
RERANK_MODEL = ".jina-reranker-v3"
QUERY = "How do distributed systems handle failures and recovery?"

query_result = es.inference.inference(
    inference_id=EMBEDDING_MODEL,
    task_type="text_embedding",
    body={"input": [QUERY]}
)
query_embedding = query_result["text_embedding"][0]["embedding"]

response = es.options(request_timeout=120).search(
    index=S3_INDEX,
    body={
        "retriever": {
            "text_similarity_reranker": {
                "retriever": {
                    "linear": {
                        "retrievers": [
                            {
                                "retriever": {
                                    "standard": {
                                        "query": {
                                            "match": {
                                                "chunk_text": QUERY
                                            }
                                        }
                                    }
                                },
                                "weight": 0.3
                            },
                            {
                                "retriever": {
                                    "knn": {
                                        "field": "embedding",
                                        "query_vector": query_embedding,
                                        "k": 10,
                                        "num_candidates": 20
                                    }
                                },
                                "weight": 0.7
                            }
                        ]
                    }
                },
                "field": "chunk_text",
                "inference_id": RERANK_MODEL,
                "inference_text": QUERY
            }
        },
        "size": 5
    },
)

print(f"\nQuery: {QUERY}")
print("Search Results (Hybrid: Linear + Reranking)")
for hit in response["hits"]["hits"]:
    src = hit["_source"]
    print(f"  Score: {hit['_score']:.4f}  Chunk #: {src['chunk_index']}")
    print(f"  Chunk: {src['chunk_text'][:60].replace(chr(10), ' ')}...")
    print(f"  ---")

## Scenario 5: External Chunking — Nested Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_text(doc_content)
print(f"Chunk count: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: {len(c.split())} words — {c[:60].replace(chr(10), ' ')}...")
print("-" * 40)

embeddings = []
for batch_start in range(0, len(chunks), 16):
    batch = chunks[batch_start:batch_start + 16]
    result = es.inference.inference(
        inference_id=EMBEDDING_MODEL,
        task_type="text_embedding",
        body={"input": batch}
    )
    embeddings.extend(item["embedding"] for item in result["text_embedding"])

S5_INDEX = "scenario5"
scenario5_mapping = {
    "mappings": {
        "properties": {
            "title": { "type": "text" },
            "chunks": {
                "type": "nested",
                "properties": {
                    "chunk_index": { "type": "integer" },
                    "chunk_text": { "type": "text" },
                    "embedding": {
                        "type": "dense_vector",
                        "dims": 1024,
                        "index": True,
                        "similarity": "cosine",
                        "index_options": { "type": "int8_hnsw" }
                    }
                }
            }
        }
    }
}

nested_chunks = [
    {"chunk_index": i, "chunk_text": c, "embedding": emb}
    for i, (c, emb) in enumerate(zip(chunks, embeddings))
]

es.options(ignore_status=[404]).indices.delete(index=S5_INDEX)
es.indices.create(index=S5_INDEX, body=scenario5_mapping)
es.index(index=S5_INDEX, document={
    "title": "Building Resilient Distributed Systems",
    "chunks": nested_chunks
})
es.indices.refresh(index=S5_INDEX)

QUERY = "How do distributed systems handle failures and recovery?"
query_result = es.inference.inference(
    inference_id=EMBEDDING_MODEL,
    task_type="text_embedding",
    body={"input": [QUERY]}
)
query_embedding = query_result["text_embedding"][0]["embedding"]

response = es.search(
    index=S5_INDEX,
    body={
        "query": {
            "nested": {
                "path": "chunks",
                "query": {
                    "knn": {
                        "field": "chunks.embedding",
                        "query_vector": query_embedding,
                        "num_candidates": 20
                    }
                },
                "inner_hits": { "size": 5, "_source": ["chunks.chunk_index", "chunks.chunk_text"] }
            }
        },
        "size": 1
    },
)

print(f"\nQuery: {QUERY}")
print("Search Results")
for hit in response["hits"]["hits"]:
    print(f"  Parent: {hit['_source']['title']}  Score: {hit['_score']:.4f}")
    for inner in hit["inner_hits"]["chunks"]["hits"]["hits"]:
        print(f"  Score: {inner['_score']:.4f}  Chunk #: {inner['_source']['chunk_index']}")
        print(f"  Chunk: {inner['_source']['chunk_text'][:60].replace(chr(10), ' ')}...")
        print(f"  ---")

## Destroy Environment

In [ ]:
%%bash
terraform -chdir=terraform destroy -auto-approve > /dev/null && echo "Done."
rm -f .env